In [1]:
!pip install langgraph langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.8 MB/s eta 0:00:00


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from google.colab import userdata

class State(TypedDict):
    task: str
    research: str
    analysis: str
    report: str

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=0,
    convert_system_message_to_human=True
)

# AGENȚI
def researcher(state):
    result = llm.invoke(f"Caută 3 informații despre: {state['task']}")
    print("✓ Researcher done")
    return {"research": result.content}

def analyst(state):
    result = llm.invoke(f"Analizează și extrage 3 insights: {state['research']}")
    print("✓ Analyst done")
    return {"analysis": result.content}

def writer(state):
    result = llm.invoke(f"Scrie raport (200 cuvinte): {state['analysis']}")
    print("✓ Writer done")
    return {"report": result.content}

# SUPERVISOR
def supervisor(state) -> Literal["research", "analyze", "write", "end"]:
    if not state.get("research"):
        return "research"
    elif not state.get("analysis"):
        return "analyze"
    elif not state.get("report"):
        return "write"
    return "end"

# GRAPH
workflow = StateGraph(State)
workflow.add_node("research", researcher)
workflow.add_node("analyze", analyst)
workflow.add_node("write", writer)

workflow.set_entry_point("research")

workflow.add_conditional_edges(
    "research",
    supervisor,
    {"analyze": "analyze", "end": END}
)

workflow.add_conditional_edges(
    "analyze",
    supervisor,
    {"write": "write", "end": END}
)

workflow.add_edge("write", END)

app = workflow.compile()

# EXECUTARE
print("🚀 Research Team Start\n")
result = app.invoke({
    "task": "Impactul AI în educație",
    "research": "",
    "analysis": "",
    "report": ""
})

print("\n📄 RAPORT FINAL:")
print("="*60)
print(result["report"])

🚀 Research Team Start

✓ Researcher done
✓ Analyst done
✓ Writer done

📄 RAPORT FINAL:
**Raport privind Impactul Inteligenței Artificiale (AI) în Educație: O Analiză a Oportunităților și Provocărilor**

Acest raport analizează trei insight-uri cheie privind impactul inteligenței artificiale (AI) în educație, evidențiind oportunitățile transformatoare și provocările inerente.

În primul rând, AI redefinește fundamental experiența educațională prin **personalizarea profundă a învățării**. Aceasta permite adaptarea conținutului și ritmului la nevoile fiecărui elev, optimizând procesul didactic. Concomitent, AI eliberează profesorii de sarcini administrative, permițându-le să se concentreze pe pedagogie, mentorat și interacțiune directă, valorificând expertiza umană.

În al doilea rând, **integrarea responsabilă a AI este condiționată de abordarea proactivă a provocărilor etice și practice**. Riscuri precum plagiatul, confidențialitatea datelor și prejudecățile algoritmice necesită cadre e